# Day 2. From a model to a measurement

**Instructor notebook.** Runs on paid Colab Pro with GPU.

We pick up where Day 1 left off. Same NTE corpus. Same off-the-shelf DeBERTa. The new things are Park's actual 13-hypothesis ensemble from Table A.1 of the RIO appendix, run with her country-year-aware premise format, and a country-year proxy plotted against known events using the published `deberta_score` column.

**Honest framing up front.** The off-the-shelf DeBERTa-v3-base used here is not the model that produced the published proxy. Park's published proxy is a DeBERTa-v3-large fine-tuned on 300 hand-coded paragraphs over 10-fold cross-validation. This notebook approximates her workflow with off-the-shelf zero-shot and benchmarks it against her published `deberta_score`. Expect a positive but imperfect correlation. Closing the gap is what fine-tuning buys.

**Before you run.** Make sure `NTE_IPR_final_v2.csv` is uploaded to your Google Drive at `MyDrive/nte_text/NTE_IPR_final_v2.csv`.

## 1. Setup

Mount Drive, install libraries, set up imports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers scikit-learn

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import pipeline
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from scipy.stats import pearsonr, spearmanr

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. Load the NTE corpus

Same CSV as Day 1. Four columns. `country` (ISO3 or full name), `year`, `text` (the IP-section paragraph), and `deberta_score` (Park's fine-tuned proxy on a roughly &minus;5 to +5 scale).

We treat `deberta_score` as the published benchmark. The off-the-shelf zero-shot scores we are about to compute will be compared to it. In a real research project, the benchmark would be hand-coded by you and a co-author.

In [ ]:
CSV_PATH = '/content/drive/MyDrive/nte_text/NTE_IPR_final_v2.csv'
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
print(f'Years: {df.year.min()} to {df.year.max()}')
print(f'Countries: {df.country.nunique()}')
print(f'Published deberta_score range: {df.deberta_score.min():+.2f} to {df.deberta_score.max():+.2f}')
df.head(3)

## 3. Build a 30-paragraph stratified sample

Sample five paragraphs from each of six year bins covering 1995-2022. This is the working set for the validation comparison. In a real project, you would replace `deberta_score` with hand-codes from your codebook. The validation logic is identical.

In [ ]:
rng = np.random.default_rng(42)
year_bins = pd.cut(df['year'], bins=6, labels=False)
sample_idx = []
for b in sorted(year_bins.unique()):
    pool = df.index[year_bins == b].tolist()
    sample_idx.extend(rng.choice(pool, size=5, replace=False).tolist())

gold = df.loc[sample_idx].reset_index(drop=True)
print(f'Sample: {len(gold)} paragraphs, {gold.country.nunique()} countries, years {gold.year.min()}-{gold.year.max()}')
print(f'Published score range in sample: {gold.deberta_score.min():+.2f} to {gold.deberta_score.max():+.2f}')

## 4. Load the off-the-shelf encoder

`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`. ~184M params, MIT license. Same model Day 1 used.

Note that Park's published proxy uses DeBERTa-v3-**large** (~435M params) fine-tuned on hand-coded hypothesis-level labels. We use the smaller off-the-shelf base model for the workshop because it loads on free Colab and runs the demo within the time budget.

In [ ]:
MODEL_ID = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
device = 0 if torch.cuda.is_available() else -1
zsc = pipeline('zero-shot-classification', model=MODEL_ID, device=device)
print(f'Loaded {MODEL_ID} on {"GPU" if device == 0 else "CPU"}')

## 5. Park's premise format

Park's appendix wraps every paragraph in a country-year-aware premise. This shim gives the NLI model the institutional context it needs for hypotheses like "the country is on the Watch List" to resolve correctly. Without it, the model has to infer the country and year from the text alone, which is fragile.

In [ ]:
def make_premise(country, year, body):
    return (f"This text is about the IPR protection situation in country "
            f"{country} and year {int(year)}: {body}")

gold['premise'] = [make_premise(c, y, t) for c, y, t in zip(gold['country'], gold['year'], gold['text'])]
print('Example premise:')
print(gold['premise'].iloc[0][:300] + '...')

## 6. Baseline. One hypothesis, three labels

Day 1's setup, applied to the country-year-aware premise. Three candidate labels with multi-class softmax. The predicted class is the argmax. We map the argmax to a sign so we can correlate it against the published proxy.

In [ ]:
baseline_labels = [
    'the country has weak IP protection and the US criticizes it',
    'the country has strong IP protection and the US praises it',
    'the paragraph is unrelated or balanced',
]
label_to_sign = {baseline_labels[0]: -1, baseline_labels[1]: +1, baseline_labels[2]: 0}

t0 = time.time()
baseline_preds = zsc(gold['premise'].tolist(), candidate_labels=baseline_labels, batch_size=8)
print(f'Baseline ran in {time.time()-t0:.1f} sec')

# continuous baseline score = P(weak) - P(strong)
def critic_minus_praise(p):
    d = dict(zip(p['labels'], p['scores']))
    return d[baseline_labels[1]] - d[baseline_labels[0]]  # +ve = praise

gold['score_baseline'] = [critic_minus_praise(p) for p in baseline_preds]
gold['pred_baseline'] = [label_to_sign[p['labels'][0]] for p in baseline_preds]

r_base, _ = pearsonr(gold['score_baseline'], gold['deberta_score'])
rho_base, _ = spearmanr(gold['score_baseline'], gold['deberta_score'])
print(f'\nBaseline (1 hypothesis) vs published proxy:')
print(f'  Pearson r  = {r_base:+.3f}')
print(f'  Spearman rho = {rho_base:+.3f}')

## 7. Park's 13 hypotheses, with weights

Direct from Table A.1 of the RIO appendix. Six factual (F) hypotheses with verifiable textual markers, seven interpretive (I) hypotheses about authorial stance. Watch-List-tied hypotheses get heavier weights because Special 301 placement is an adjudicated USTR process.

In [ ]:
hypotheses = [
    ('H1',  'The country is the Priority Foreign Country.', -2.0, 'factual'),
    ('H2',  'The country is on the Priority Watch List.', -2.0, 'factual'),
    ('H3',  'The country is on the Watch List.', -1.5, 'factual'),
    ('H4',  'The country has markets listed as the Notorious Market.', -1.5, 'factual'),
    ('H5',  'The author of this text believes that the country does not put in efforts to combat IP violations.', -1.0, 'interpretive'),
    ('H6',  'The author of this text believes that the country has made efforts to combat IP violations.',  +1.0, 'interpretive'),
    ('H7',  'The author of this text supports the passage of the new IP legislation in the country.',       +1.0, 'interpretive'),
    ('H8',  'The author of this text opposes the passage of the new IP legislation in the country.',       -1.0, 'interpretive'),
    ('H9',  'The author of this text believes that there is widespread IP violation in the country.',      -1.5, 'interpretive'),
    ('H10', 'The author of this text believes that the country is lack of resources to combat IP violations.', -1.0, 'interpretive'),
    ('H11', 'The author of this text believes that the country has strong IP law.',                         +2.0, 'interpretive'),
    ('H12', 'This text mentions the increase of IP violations in the country.', -1.0, 'factual'),
    ('H13', 'This text mentions the decrease of IP violations in the country.', +1.0, 'factual'),
]
hyp_ids = [h[0] for h in hypotheses]
hyp_texts = [h[1] for h in hypotheses]
hyp_weights = np.array([h[2] for h in hypotheses])
hyp_types = [h[3] for h in hypotheses]
print(f'{len(hypotheses)} hypotheses defined.')
print(f'Sum of positive weights: {hyp_weights[hyp_weights>0].sum():+.1f}')
print(f'Sum of negative weights: {hyp_weights[hyp_weights<0].sum():+.1f}')
print(f'Asymmetry intentional. Park has more negative hypotheses because the NTE genre is dominated by US criticism.')

## 8. Run the ensemble on the 30-paragraph sample

Each premise is checked against each hypothesis with `multi_label=True`, so each hypothesis returns its own entailment probability rather than competing on a softmax. The continuous score is the weighted sum, normalized by the sum of absolute weights.

In [ ]:
def ensemble_scores(premises, hyp_texts, hyp_weights, batch_size=8):
    preds = zsc(premises, candidate_labels=hyp_texts, multi_label=True, batch_size=batch_size)
    weight_norm = np.abs(hyp_weights).sum()
    out = []
    for p in preds:
        prob_by_hyp = dict(zip(p['labels'], p['scores']))
        probs = np.array([prob_by_hyp[h] for h in hyp_texts])
        out.append(float((probs * hyp_weights).sum() / weight_norm))
    return out

t0 = time.time()
gold['score_ensemble'] = ensemble_scores(gold['premise'].tolist(), hyp_texts, hyp_weights)
print(f'Ensemble on {len(gold)} paragraphs ran in {time.time()-t0:.1f} sec')
print(f'Score range: {min(gold["score_ensemble"]):+.3f} to {max(gold["score_ensemble"]):+.3f}')

## 9. Compare baseline vs ensemble against the published proxy

Both off-the-shelf approaches are benchmarked against `deberta_score`. The ensemble is expected to track Park's proxy more closely than the single-hypothesis baseline because it covers more of the conceptual surface.

In [ ]:
r_ens, _ = pearsonr(gold['score_ensemble'], gold['deberta_score'])
rho_ens, _ = spearmanr(gold['score_ensemble'], gold['deberta_score'])

print(f'Off-the-shelf zero-shot vs Park\'s published proxy:')
print(f'  {"approach":<30} {"Pearson r":>10} {"Spearman rho":>14}')
print('-' * 60)
print(f'  {"baseline (1 hypothesis)":<30} {r_base:>+10.3f} {rho_base:>+14.3f}')
print(f'  {"ensemble (Park 13 hypotheses)":<30} {r_ens:>+10.3f} {rho_ens:>+14.3f}')
print()
print(f'Park\'s published cross-validation numbers, for reference (from RIO appendix):')
print(f'  fine-tuned DeBERTa-v3-large    Pearson r = +0.676   Spearman rho = +0.689')
print(f'\nThe gap between off-the-shelf ensemble and fine-tuned is what 300 hand-coded paragraphs')
print(f'and 10 folds of training buy you. Even off-the-shelf hypothesis engineering moves in the right direction.')

## 10. Inspect the disagreements

Where does off-the-shelf zero-shot disagree most with Park's fine-tuned proxy? These are the paragraphs that motivate fine-tuning.

In [ ]:
# Rescale ensemble score to roughly the same range as deberta_score for direct comparison
ens = gold['score_ensemble'].values
rescaled = -5 + 10 * (ens - ens.min()) / (ens.max() - ens.min() + 1e-9)
gold['score_ensemble_rescaled'] = rescaled
gold['gap'] = gold['score_ensemble_rescaled'] - gold['deberta_score']

worst = gold.reindex(gold['gap'].abs().sort_values(ascending=False).index).head(5)
for _, r in worst.iterrows():
    print(f'\n[{r.country} {int(r.year)}]  published={r.deberta_score:+.2f}  ensemble~={r.score_ensemble_rescaled:+.2f}  gap={r.gap:+.2f}')
    print('  ' + str(r.text)[:280] + ('...' if len(str(r.text)) > 280 else ''))

## 11. Country-year aggregation from the published proxy

Now switch from validation to plotting. Use the published `deberta_score` directly (not the off-the-shelf ensemble) so the plot reflects what Park reports. We compute country-year means and counts from the full corpus.

In [ ]:
cy = (df.groupby(['country','year'])
        .agg(score=('deberta_score','mean'),
             n_paragraphs=('deberta_score','size'))
        .reset_index())
print(f'{len(cy)} country-year observations')
print(f'Score range: {cy.score.min():+.2f} to {cy.score.max():+.2f}')
print(cy.head(8).to_string(index=False))

## 12. Plot Israel and Turkey against external events

Israel was removed from the Special 301 Report in 2014. Turkey faced a coup attempt in mid-2016 followed by sustained democratic erosion. Both events are external. Neither is hand-engineered into the model. The proxy should show both.

In [ ]:
case_lookup = {'Israel': 'ISRAEL', 'Turkey': 'TURKEY'}
fig, ax = plt.subplots(figsize=(11, 4.5))

colors = {'Israel': '#334B99', 'Turkey': '#2D493C'}
for label, code in case_lookup.items():
    sub = cy[cy['country'].str.upper() == code].sort_values('year')
    if len(sub) == 0:
        print(f'No rows for {label} ({code}).')
        continue
    ax.plot(sub['year'], sub['score'], marker='o', label=label, color=colors[label], linewidth=2.4, markersize=4)

ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.axvline(2014, color='#334B99', linewidth=1.5, linestyle='--', alpha=0.7)
ax.axvline(2016, color='#334B99', linewidth=1.5, linestyle='--', alpha=0.7)
ax.text(2014, ax.get_ylim()[1]*1.02, '2014', fontsize=10, color='#334B99', fontweight='bold', ha='center')
ax.text(2016, ax.get_ylim()[1]*1.02, '2016', fontsize=10, color='#334B99', fontweight='bold', ha='center')
ax.set_xlabel('Year')
ax.set_ylabel('Country-year proxy score')
ax.set_title('Park\'s published proxy plotted against external events')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 13. Save the proxy CSV

What the rest of the paper plugs into. One row per country-year. Always keep the paragraph count for caveats.

In [ ]:
OUT_PATH = '/content/drive/MyDrive/nte_text/nte_proxy_country_year.csv'
cy.to_csv(OUT_PATH, index=False)
print(f'Wrote {len(cy)} rows to {OUT_PATH}')

## What we just did

1. Ran an off-the-shelf single-hypothesis baseline against the country-year-aware premise format. Pearson r against the published proxy was the baseline number.
2. Implemented Park's 13-hypothesis ensemble from Table A.1 with her exact weights.
3. Compared the off-the-shelf ensemble against the published fine-tuned proxy. Saw a positive correlation that does not reach the published Pearson r of 0.676. The gap is what fine-tuning closes.
4. Aggregated the published proxy to country-year and plotted Israel and Turkey against external events.
5. Saved the country-year CSV.

**Day 3** introduces generative models. Same reproducibility demands. Different toolkit. Useful for tasks the encoder cannot easily express as a label.